# Using a DB in sqlite3


## SQL in sqlite3

In [2]:
import sqlite3 as db
import pandas as pd
import contextlib


In [3]:
%%capture
%%bash
apt-get update
apt-get install -y sqlite3


In [ ]:
!sqlite3 --help


In [ ]:
!sqlite3 --version

## Sample database

The [Chinook sample](https://www.sqlitetutorial.net/sqlite-sample-database/)

- Zip file of [sqlite DB](https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip)

- [ER diagram](https://www.sqlitetutorial.net/wp-content/uploads/2018/03/sqlite-sample-database-diagram-color.pdf)

- [ER symbols/meanings](https://d2slcw3kip6qmk.cloudfront.net/marketing/pages/chart/erd-symbols/ERD-Notation.PNG)

- [Draw IO](https://app.diagrams.net/) for creating ER diagrams, flowcharts, etc.

In [4]:
%%bash
[ -f chinook.zip ] ||
  curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
unzip -l chinook.zip


Archive:  chinook.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   884736  2015-11-29 10:53   chinook.db
---------                     -------
   884736                     1 file


In [5]:
!rm -f chinook.db


In [6]:
!unzip -u chinook.zip


Archive:  chinook.zip
  inflating: chinook.db              


In [7]:
!ls -l

total 1168
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 Nov 10 21:44 chinook.zip
drwxr-xr-x 1 root root   4096 Oct 26 22:18 sample_data


## Query using Python


### Without using a context manager


In [8]:
# Connect to a sqlite database and load a result set into a data frame
db_con = db.connect("chinook.db")


In [9]:
# Select the first 10 entries from the employees table
query = '''
  select *
  from employees
  limit 10
  ;
'''
print(query)


  select *
  from employees
  limit 10
  ;



In [10]:
employees = pd.read_sql_query( query , db_con)
employees

,EmployeeId,LastName,FirstName,Title,ReportsTo,BirthDate,HireDate,Address,City,State,Country,PostalCode,Phone,Fax,Email
0,1,Adams,Andrew,General Manager,NaN,1962-02-18 00:00:00,2002-08-14 00:00:00,11120 Jasper Ave NW,Edmonton,AB,Canada,T5K 2N1,+1 (780) 428-9482,+1 (780) 428-3457,andrew@chinookcorp.com
1,2,Edwards,Nancy,Sales Manager,1.0,1958-12-08 00:00:00,2002-05-01 00:00:00,825 8 Ave SW,Calgary,AB,Canada,T2P 2T3,+1 (403) 262-3443,+1 (403) 262-3322,nancy@chinookcorp.com
2,3,Peacock,Jane,Sales Support Agent,2.0,1973-08-29 00:00:00,2002-04-01 00:00:00,1111 6 Ave SW,Calgary,AB,Canada,T2P 5M5,+1 (403) 262-3443,+1 (403) 262-6712,jane@chinookcorp.com
3,4,Park,Margaret,Sales Support Agent,2.0,1947-09-19 00:00:00,2003-05-03 00:00:00,683 10 Street SW,Calgary,AB,Canada,T2P 5G3,+1 (403) 263-4423,+1 (403) 263-4289,margaret@chinookcorp.com
4,5,Johnson,Steve,Sales Support Agent,2.0,1965-03-03 00:00:00,2003-10-17 00:00:00,7727B 41 Ave,Calgary,AB,Canada,T3B 1Y7,1 (780) 836-9987,1 (780) 836-9543,steve@chinookcorp.com
5,6,Mitchell,Michael,IT Manager,1.0,1973-07-01 00:00:00,2003-10-17 00:00:00,5827 Bowness Road NW,Calgary,AB,Canada,T3B 0C5,+1 (403) 246-9887,+1 (403) 246-9899,michael@chinookcorp.com
6,7,King,Robert,IT Staff,6.0,1970-05-29 00:00:00,2004-01-02 00:00:00,590 Columbia Boulevard West,Lethbridge,AB,Canada,T1K 5N8,+1 (403) 456-9986,+1 (403) 456-8485,robert@chinookcorp.com
7,8,Callahan,Laura,IT Staff,6.0,1968-01-09 00:00:00,2004-03-04 00:00:00,923 7 ST NW,Lethbridge,AB,Canada,T1H 1Y8,+1 (403) 467-3351,+1 (403) 467-8772,laura@chinookcorp.com


In [11]:
employees.shape


(8, 15)

In [12]:
employees.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   EmployeeId  8 non-null      int64  
 1   LastName    8 non-null      object 
 2   FirstName   8 non-null      object 
 3   Title       8 non-null      object 
 4   ReportsTo   7 non-null      float64
 5   BirthDate   8 non-null      object 
 6   HireDate    8 non-null      object 
 7   Address     8 non-null      object 
 8   City        8 non-null      object 
 9   State       8 non-null      object 
 10  Country     8 non-null      object 
 11  PostalCode  8 non-null      object 
 12  Phone       8 non-null      object 
 13  Fax         8 non-null      object 
 14  Email       8 non-null      object 
dtypes: float64(1), int64(1), object(13)
memory usage: 1.1+ KB


In [13]:
# explicitly close the connection
db_con.close()

### Using a context manager

List of all table in the database using SQL

In [14]:
query = '''
  SELECT
    name
  FROM
    sqlite_master
  WHERE
    type='table' AND
    name not like "sqlite_%"
;
'''
print(query)


  SELECT
    name
  FROM
    sqlite_master
  WHERE
    type='table' AND
    name not like "sqlite_%"
;



In [15]:
with contextlib.closing(db.connect("chinook.db")) as db_con:
  tables = pd.read_sql_query( query , db_con)

tables


,name
0,albums
1,artists
2,customers
3,employees
4,genres
5,invoices
6,invoice_items
7,media_types
8,playlists
9,playlist_track


In [26]:
rows = []
with contextlib.closing(db.connect("chinook.db")) as db_con:
  for table in tables["name"]:
    # print(f"== {table} row count ==> ", end='')
    query = f'''select count(*) as row_count from {table} ;'''
    row_count = pd.read_sql_query( query, db_con ).iloc[0, 0]
    # print(row_count)
    rows.append([table, row_count])

rows_df = pd.DataFrame( rows, columns = [ "table", "row_count" ] )
rows_df.sort_values( by = ["row_count"], ascending=False )

,table,row_count
9,playlist_track,8715
10,tracks,3503
6,invoice_items,2240
5,invoices,412
0,albums,347
1,artists,275
2,customers,59
4,genres,25
8,playlists,18
3,employees,8


In [27]:
# Select the first 10 entries from the employees table
query = '''
  select *
  from employees
  limit 10
'''
print(query)


  select *
  from employees
  limit 10



In [28]:
# Connect to a sqlite database, load a result set into a data frame, and close the connection
with contextlib.closing(db.connect("chinook.db")) as db_con:
  employees = pd.read_sql_query( query , db_con)

employees

,EmployeeId,LastName,FirstName,Title,ReportsTo,BirthDate,HireDate,Address,City,State,Country,PostalCode,Phone,Fax,Email
0,1,Adams,Andrew,General Manager,NaN,1962-02-18 00:00:00,2002-08-14 00:00:00,11120 Jasper Ave NW,Edmonton,AB,Canada,T5K 2N1,+1 (780) 428-9482,+1 (780) 428-3457,andrew@chinookcorp.com
1,2,Edwards,Nancy,Sales Manager,1.0,1958-12-08 00:00:00,2002-05-01 00:00:00,825 8 Ave SW,Calgary,AB,Canada,T2P 2T3,+1 (403) 262-3443,+1 (403) 262-3322,nancy@chinookcorp.com
2,3,Peacock,Jane,Sales Support Agent,2.0,1973-08-29 00:00:00,2002-04-01 00:00:00,1111 6 Ave SW,Calgary,AB,Canada,T2P 5M5,+1 (403) 262-3443,+1 (403) 262-6712,jane@chinookcorp.com
3,4,Park,Margaret,Sales Support Agent,2.0,1947-09-19 00:00:00,2003-05-03 00:00:00,683 10 Street SW,Calgary,AB,Canada,T2P 5G3,+1 (403) 263-4423,+1 (403) 263-4289,margaret@chinookcorp.com
4,5,Johnson,Steve,Sales Support Agent,2.0,1965-03-03 00:00:00,2003-10-17 00:00:00,7727B 41 Ave,Calgary,AB,Canada,T3B 1Y7,1 (780) 836-9987,1 (780) 836-9543,steve@chinookcorp.com
5,6,Mitchell,Michael,IT Manager,1.0,1973-07-01 00:00:00,2003-10-17 00:00:00,5827 Bowness Road NW,Calgary,AB,Canada,T3B 0C5,+1 (403) 246-9887,+1 (403) 246-9899,michael@chinookcorp.com
6,7,King,Robert,IT Staff,6.0,1970-05-29 00:00:00,2004-01-02 00:00:00,590 Columbia Boulevard West,Lethbridge,AB,Canada,T1K 5N8,+1 (403) 456-9986,+1 (403) 456-8485,robert@chinookcorp.com
7,8,Callahan,Laura,IT Staff,6.0,1968-01-09 00:00:00,2004-03-04 00:00:00,923 7 ST NW,Lethbridge,AB,Canada,T1H 1Y8,+1 (403) 467-3351,+1 (403) 467-8772,laura@chinookcorp.com


In [29]:
employees.shape


(8, 15)

In [30]:
employees.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   EmployeeId  8 non-null      int64  
 1   LastName    8 non-null      object 
 2   FirstName   8 non-null      object 
 3   Title       8 non-null      object 
 4   ReportsTo   7 non-null      float64
 5   BirthDate   8 non-null      object 
 6   HireDate    8 non-null      object 
 7   Address     8 non-null      object 
 8   City        8 non-null      object 
 9   State       8 non-null      object 
 10  Country     8 non-null      object 
 11  PostalCode  8 non-null      object 
 12  Phone       8 non-null      object 
 13  Fax         8 non-null      object 
 14  Email       8 non-null      object 
dtypes: float64(1), int64(1), object(13)
memory usage: 1.1+ KB


In [31]:
query


'\n  select *\n  from employees\n  limit 10\n'

This doesn't work b/c the db is closed:

In [ ]:
#pd.read_sql_query( query , db_con)

Reopen and one-shot, then close:

In [32]:
with contextlib.closing(db.connect("chinook.db")) as db_con:
  display(pd.read_sql_query( query , db_con))

,EmployeeId,LastName,FirstName,Title,ReportsTo,BirthDate,HireDate,Address,City,State,Country,PostalCode,Phone,Fax,Email
0,1,Adams,Andrew,General Manager,NaN,1962-02-18 00:00:00,2002-08-14 00:00:00,11120 Jasper Ave NW,Edmonton,AB,Canada,T5K 2N1,+1 (780) 428-9482,+1 (780) 428-3457,andrew@chinookcorp.com
1,2,Edwards,Nancy,Sales Manager,1.0,1958-12-08 00:00:00,2002-05-01 00:00:00,825 8 Ave SW,Calgary,AB,Canada,T2P 2T3,+1 (403) 262-3443,+1 (403) 262-3322,nancy@chinookcorp.com
2,3,Peacock,Jane,Sales Support Agent,2.0,1973-08-29 00:00:00,2002-04-01 00:00:00,1111 6 Ave SW,Calgary,AB,Canada,T2P 5M5,+1 (403) 262-3443,+1 (403) 262-6712,jane@chinookcorp.com
3,4,Park,Margaret,Sales Support Agent,2.0,1947-09-19 00:00:00,2003-05-03 00:00:00,683 10 Street SW,Calgary,AB,Canada,T2P 5G3,+1 (403) 263-4423,+1 (403) 263-4289,margaret@chinookcorp.com
4,5,Johnson,Steve,Sales Support Agent,2.0,1965-03-03 00:00:00,2003-10-17 00:00:00,7727B 41 Ave,Calgary,AB,Canada,T3B 1Y7,1 (780) 836-9987,1 (780) 836-9543,steve@chinookcorp.com
5,6,Mitchell,Michael,IT Manager,1.0,1973-07-01 00:00:00,2003-10-17 00:00:00,5827 Bowness Road NW,Calgary,AB,Canada,T3B 0C5,+1 (403) 246-9887,+1 (403) 246-9899,michael@chinookcorp.com
6,7,King,Robert,IT Staff,6.0,1970-05-29 00:00:00,2004-01-02 00:00:00,590 Columbia Boulevard West,Lethbridge,AB,Canada,T1K 5N8,+1 (403) 456-9986,+1 (403) 456-8485,robert@chinookcorp.com
7,8,Callahan,Laura,IT Staff,6.0,1968-01-09 00:00:00,2004-03-04 00:00:00,923 7 ST NW,Lethbridge,AB,Canada,T1H 1Y8,+1 (403) 467-3351,+1 (403) 467-8772,laura@chinookcorp.com


## Query using SQLite3

Using Jupyter magics

In [33]:
# Get a list of the tables in the database
%%script sqlite3 --column --header chinook.db
.tables


albums          employees       invoices        playlists     
artists         genres          media_types     tracks        
customers       invoice_items   playlist_track


In [34]:
# Show the schema for the entire database
%%script sqlite3 --column --header chinook.db
.schema


CREATE TABLE IF NOT EXISTS "albums"
(
    [AlbumId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Title] NVARCHAR(160)  NOT NULL,
    [ArtistId] INTEGER  NOT NULL,
    FOREIGN KEY ([ArtistId]) REFERENCES "artists" ([ArtistId]) 
		ON DELETE NO ACTION ON UPDATE NO ACTION
);
CREATE TABLE sqlite_sequence(name,seq);
CREATE TABLE IF NOT EXISTS "artists"
(
    [ArtistId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);
CREATE TABLE IF NOT EXISTS "customers"
(
    [CustomerId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [FirstName] NVARCHAR(40)  NOT NULL,
    [LastName] NVARCHAR(20)  NOT NULL,
    [Company] NVARCHAR(80),
    [Address] NVARCHAR(70),
    [City] NVARCHAR(40),
    [State] NVARCHAR(40),
    [Country] NVARCHAR(40),
    [PostalCode] NVARCHAR(10),
    [Phone] NVARCHAR(24),
    [Fax] NVARCHAR(24),
    [Email] NVARCHAR(60)  NOT NULL,
    [SupportRepId] INTEGER,
    FOREIGN KEY ([SupportRepId]) REFERENCES "employees" ([EmployeeId]) 
		ON DELETE NO ACTION ON 

In [35]:
# Select the first 10 entries from the employees table
%%script sqlite3 --column --header chinook.db
select *
from employees
limit 10


EmployeeId  LastName  FirstName  Title                ReportsTo  BirthDate            HireDate             Address                      City        State  Country  PostalCode  Phone              Fax                Email                   
----------  --------  ---------  -------------------  ---------  -------------------  -------------------  ---------------------------  ----------  -----  -------  ----------  -----------------  -----------------  ------------------------
1           Adams     Andrew     General Manager                 1962-02-18 00:00:00  2002-08-14 00:00:00  11120 Jasper Ave NW          Edmonton    AB     Canada   T5K 2N1     +1 (780) 428-9482  +1 (780) 428-3457  andrew@chinookcorp.com  
2           Edwards   Nancy      Sales Manager        1          1958-12-08 00:00:00  2002-05-01 00:00:00  825 8 Ave SW                 Calgary     AB     Canada   T2P 2T3     +1 (403) 262-3443  +1 (403) 262-3322  nancy@chinookcorp.com   
3           Peacock   Jane       Sales Suppo

In [36]:
# Count the number of unique albums
%%script sqlite3 --column --header chinook.db
select count(distinct AlbumID)
from tracks


count(distinct AlbumID)
-----------------------
347                    


In [38]:
# Count the number of albums in the tracks table
%%script sqlite3 --column --header chinook.db
select count(AlbumID)
from tracks

count(AlbumID)
--------------
3503          


In [39]:
# Count the number of albums in the albums table
%%script sqlite3 --column --header chinook.db
select count(AlbumID)
from albums

count(AlbumID)
--------------
347           


In [40]:
# Select the first 10 entries from the albums table
%%script sqlite3 --column --header chinook.db
select *
from albums
limit 10


AlbumId  Title                                  ArtistId
-------  -------------------------------------  --------
1        For Those About To Rock We Salute You  1       
2        Balls to the Wall                      2       
3        Restless and Wild                      2       
4        Let There Be Rock                      1       
5        Big Ones                               3       
6        Jagged Little Pill                     4       
7        Facelift                               5       
8        Warner 25 Anos                         6       
9        Plays Metallica By Four Cellos         7       
10       Audioslave                             8       


In [41]:
# Select the first 10 entries from the artists table
%%script sqlite3 --column --header chinook.db
select *
from artists
limit 10


ArtistId  Name                
--------  --------------------
1         AC/DC               
2         Accept              
3         Aerosmith           
4         Alanis Morissette   
5         Alice In Chains     
6         Antônio Carlos Jobim
7         Apocalyptica        
8         Audioslave          
9         BackBeat            
10        Billy Cobham        


In [42]:
# Select the first 10 entries from joining the artists and albums tables
%%script sqlite3 --column --header chinook.db
select *
from artists
join albums
on artists.ArtistID = albums.ArtistID
limit 10


ArtistId  Name                  AlbumId  Title                                  ArtistId
--------  --------------------  -------  -------------------------------------  --------
1         AC/DC                 1        For Those About To Rock We Salute You  1       
2         Accept                2        Balls to the Wall                      2       
2         Accept                3        Restless and Wild                      2       
1         AC/DC                 4        Let There Be Rock                      1       
3         Aerosmith             5        Big Ones                               3       
4         Alanis Morissette     6        Jagged Little Pill                     4       
5         Alice In Chains       7        Facelift                               5       
6         Antônio Carlos Jobim  8        Warner 25 Anos                         6       
7         Apocalyptica          9        Plays Metallica By Four Cellos         7       
8         Audioslave 

# HERE

In [43]:
# Select albums by AC/DC
%%script sqlite3 --column --header chinook.db
select *
from artists
join albums
on artists.ArtistID = albums.ArtistID
where Name = 'AC/DC'
limit 10


ArtistId  Name   AlbumId  Title                                  ArtistId
--------  -----  -------  -------------------------------------  --------
1         AC/DC  1        For Those About To Rock We Salute You  1       
1         AC/DC  4        Let There Be Rock                      1       


In [44]:
# Select some fields from the employees table
%%script sqlite3 --column --header chinook.db
select EmployeeId, LastName, FirstName, Title, ReportsTo
from employees


EmployeeId  LastName  FirstName  Title                ReportsTo
----------  --------  ---------  -------------------  ---------
1           Adams     Andrew     General Manager               
2           Edwards   Nancy      Sales Manager        1        
3           Peacock   Jane       Sales Support Agent  2        
4           Park      Margaret   Sales Support Agent  2        
5           Johnson   Steve      Sales Support Agent  2        
6           Mitchell  Michael    IT Manager           1        
7           King      Robert     IT Staff             6        
8           Callahan  Laura      IT Staff             6        


In [45]:
# Select employees and their bosses
%%script sqlite3 --column --header chinook.db
select emp1.EmployeeId, emp1.LastName, emp1.FirstName, emp1.Title, emp1.ReportsTo, emp2.EmployeeId, emp2.LastName, emp2.FirstName, emp2.ReportsTo
from employees as emp1
join employees as emp2
on emp1.ReportsTo = emp2.EmployeeID


EmployeeId  LastName  FirstName  Title                ReportsTo  EmployeeId  LastName  FirstName  ReportsTo
----------  --------  ---------  -------------------  ---------  ----------  --------  ---------  ---------
2           Edwards   Nancy      Sales Manager        1          1           Adams     Andrew              
3           Peacock   Jane       Sales Support Agent  2          2           Edwards   Nancy      1        
4           Park      Margaret   Sales Support Agent  2          2           Edwards   Nancy      1        
5           Johnson   Steve      Sales Support Agent  2          2           Edwards   Nancy      1        
6           Mitchell  Michael    IT Manager           1          1           Adams     Andrew              
7           King      Robert     IT Staff             6          6           Mitchell  Michael    1        
8           Callahan  Laura      IT Staff             6          6           Mitchell  Michael    1        


In [46]:
# The first three entries from a self cross join of employees
%%script sqlite3 --column --header chinook.db
select *
from employees as emp1
join employees as emp2
limit 3

EmployeeId  LastName  FirstName  Title            ReportsTo  BirthDate            HireDate             Address              City      State  Country  PostalCode  Phone              Fax                Email                   EmployeeId  LastName  FirstName  Title                ReportsTo  BirthDate            HireDate             Address              City      State  Country  PostalCode  Phone              Fax                Email                 
----------  --------  ---------  ---------------  ---------  -------------------  -------------------  -------------------  --------  -----  -------  ----------  -----------------  -----------------  ----------------------  ----------  --------  ---------  -------------------  ---------  -------------------  -------------------  -------------------  --------  -----  -------  ----------  -----------------  -----------------  ----------------------
1           Adams     Andrew     General Manager             1962-02-18 00:00:00  2002-08-14 00:00

In [47]:
# Select a few columns from a self cross joins of employees
%%script sqlite3 --column --header chinook.db
select emp1.EmployeeID, emp1.FirstName, emp1.ReportsTo, emp2.EmployeeID, emp2.FirstName
from employees as emp1
join employees as emp2


EmployeeId  FirstName  ReportsTo  EmployeeId  FirstName
----------  ---------  ---------  ----------  ---------
1           Andrew                1           Andrew   
1           Andrew                2           Nancy    
1           Andrew                3           Jane     
1           Andrew                4           Margaret 
1           Andrew                5           Steve    
1           Andrew                6           Michael  
1           Andrew                7           Robert   
1           Andrew                8           Laura    
2           Nancy      1          1           Andrew   
2           Nancy      1          2           Nancy    
2           Nancy      1          3           Jane     
2           Nancy      1          4           Margaret 
2           Nancy      1          5           Steve    
2           Nancy      1          6           Michael  
2           Nancy      1          7           Robert   
2           Nancy      1          8           La

In [48]:
# Select the first 10 entries from a self cross joins of employees
%%script sqlite3 --column --header chinook.db
select emp1.EmployeeID, emp1.FirstName, emp1.ReportsTo, emp2.EmployeeID, emp2.FirstName
from employees as emp1
join employees as emp2
limit 10

EmployeeId  FirstName  ReportsTo  EmployeeId  FirstName
----------  ---------  ---------  ----------  ---------
1           Andrew                1           Andrew   
1           Andrew                2           Nancy    
1           Andrew                3           Jane     
1           Andrew                4           Margaret 
1           Andrew                5           Steve    
1           Andrew                6           Michael  
1           Andrew                7           Robert   
1           Andrew                8           Laura    
2           Nancy      1          1           Andrew   
2           Nancy      1          2           Nancy    


In [49]:
# Select employees and their supervisors
%%script sqlite3 --column --header chinook.db
select emp1.EmployeeID, emp1.FirstName, emp1.ReportsTo, emp2.EmployeeID, emp2.FirstName
from employees as emp1
join employees as emp2
on emp1.ReportsTo = emp2.EmployeeID
order by emp1.EmployeeID

EmployeeId  FirstName  ReportsTo  EmployeeId  FirstName
----------  ---------  ---------  ----------  ---------
2           Nancy      1          1           Andrew   
3           Jane       2          2           Nancy    
4           Margaret   2          2           Nancy    
5           Steve      2          2           Nancy    
6           Michael    1          1           Andrew   
7           Robert     6          6           Michael  
8           Laura      6          6           Michael  


### Building a complex join using a CTE


In [50]:
# Select customers, their rep, and the rep's supervisor
## uses CTEs
%%script sqlite3 --column --header chinook.db

with
emp as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from employees
)

select *
from emp
limit 10


EmployeeID  FirstName  Lastname  ReportsTo
----------  ---------  --------  ---------
1           Andrew     Adams              
2           Nancy      Edwards   1        
3           Jane       Peacock   2        
4           Margaret   Park      2        
5           Steve      Johnson   2        
6           Michael    Mitchell  1        
7           Robert     King      6        
8           Laura      Callahan  6        


In [51]:
# Select customers, their rep, and the rep's supervisor
## uses CTEs
%%script sqlite3 --column --header chinook.db

with
emp as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from employees
),
reps as (
  select *
  from emp
)

select *
from reps


EmployeeID  FirstName  Lastname  ReportsTo
----------  ---------  --------  ---------
1           Andrew     Adams              
2           Nancy      Edwards   1        
3           Jane       Peacock   2        
4           Margaret   Park      2        
5           Steve      Johnson   2        
6           Michael    Mitchell  1        
7           Robert     King      6        
8           Laura      Callahan  6        


In [52]:
# Select customers, their rep, and the rep's supervisor
## uses CTEs
%%script sqlite3 --column --header chinook.db

with
emp as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from employees
),
reps as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from emp
),
supervisors as (
  select EmployeeID, FirstName, Lastname
  from emp
)

select *
from supervisors



EmployeeID  FirstName  Lastname
----------  ---------  --------
1           Andrew     Adams   
2           Nancy      Edwards 
3           Jane       Peacock 
4           Margaret   Park    
5           Steve      Johnson 
6           Michael    Mitchell
7           Robert     King    
8           Laura      Callahan


In [53]:
# Select customers, their rep, and the rep's supervisor
## uses CTEs
%%script sqlite3 --column --header chinook.db

with
emp as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from employees
),
reps as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from emp
),
supervisors as (
  select EmployeeID, FirstName, Lastname
  from emp
),
workers as (
  select r.EmployeeID as "EmpID", r.Firstname as "Rep", r.ReportsTo as "ReportID", s.EmployeeID as "SuperID", s.FirstName as "Supervisor"
  from reps as r
  inner join supervisors as s
  on r.ReportsTo = s.EmployeeID
)

select *
from workers




EmpID  Rep       ReportID  SuperID  Supervisor
-----  --------  --------  -------  ----------
2      Nancy     1         1        Andrew    
3      Jane      2         2        Nancy     
4      Margaret  2         2        Nancy     
5      Steve     2         2        Nancy     
6      Michael   1         1        Andrew    
7      Robert    6         6        Michael   
8      Laura     6         6        Michael   


In [54]:
# Select customers, their rep, and the rep's supervisor
## uses CTEs
%%script sqlite3 --column --header chinook.db

with
emp as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from employees
),
reps as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from emp
),
supervisors as (
  select EmployeeID, FirstName, Lastname
  from emp
),
workers as (
  select r.EmployeeID as "EmpID", r.Firstname as "Rep", r.ReportsTo as "ReportID", s.EmployeeID as "SuperID", s.FirstName as "Supervisor"
  from reps as r
  join supervisors as s
  where r.ReportsTo = s.EmployeeID
),
cust as (
  select CustomerID, FirstName as "Customer", SupportRepID
  from customers
)

select *
from cust
limit 10
;

CustomerID  Customer   SupportRepID
----------  ---------  ------------
1           Luís       3           
2           Leonie     5           
3           François   3           
4           Bjørn      4           
5           František  4           
6           Helena     5           
7           Astrid     5           
8           Daan       4           
9           Kara       4           
10          Eduardo    4           


In [55]:
# Select customers, their rep, and the rep's supervisor
## uses CTEs
%%script sqlite3 --column --header chinook.db

with
emp as (
  select EmployeeID, FirstName, Lastname, ReportsTo, Title
  from employees
),
reps as (
  select EmployeeID, FirstName, Lastname, ReportsTo
  from emp
  where Title = "Sales Support Agent"
),
supervisors as (
  select EmployeeID, FirstName, Lastname
  from emp
  where Title != "Sales Support Agent"
),
workers as (
  select r.EmployeeID as "EmpID", r.Firstname as "Rep", r.ReportsTo as "ReportID", s.EmployeeID as "SuperID", s.FirstName as "Supervisor"
  from reps as r
  join supervisors as s
  where r.ReportsTo = s.EmployeeID
),
cust as (
  select CustomerID, FirstName as "Customer", SupportRepID
  from customers
),
cust_rep_super as (
  select Customer, Rep, Supervisor
  -- SELECT *
  from cust
  inner join workers
  on SupportRepID = EmpID
)

SELECT *
FROM cust_rep_super
limit 10


Customer  Rep   Supervisor
--------  ----  ----------
Luís      Jane  Nancy     
François  Jane  Nancy     
Roberto   Jane  Nancy     
Jennifer  Jane  Nancy     
Michelle  Jane  Nancy     
Tim       Jane  Nancy     
Frank     Jane  Nancy     
Robert    Jane  Nancy     
Edward    Jane  Nancy     
Ellie     Jane  Nancy     


## Your Turn

- How many unique albums are in the Tracks table?
- How many tracks are on the album with AlbumID 1?
- Do the totals in the Invoices table add up to the lineitems in the Invoice_Items table?

Don't forget to add this magic to the top of your code cell or use Python with a sqlite connection.

In [ ]:
%%script sqlite3 --column --header chinook.db

... or ...

In [ ]:
import sqlite3 as db
import pandas as pd

query = '''
  SELECT ...
'''

with contextlib.closing(db.connect("chinook.db")) as db_con:
  df = pd.read_sql_query( query , db_con )
df


In [57]:
# 1. How many unique albums are in the Tracks table?
%%script sqlite3 --column --header chinook.db
select count(distinct AlbumID)
from tracks

count(distinct AlbumID)
-----------------------
347                    


In [64]:
# 2. How many tracks are on the album with AlbumID 1?
%%script sqlite3 --column --header chinook.db
select trackid, name as title
from tracks
where AlbumId = 1

TrackId  title                                  
-------  ---------------------------------------
1        For Those About To Rock (We Salute You)
6        Put The Finger On You                  
7        Let's Get It Up                        
8        Inject The Venom                       
9        Snowballed                             
10       Evil Walks                             
11       C.O.D.                                 
12       Breaking The Rules                     
13       Night Of The Long Knives               
14       Spellbound                             


In [61]:
%%script sqlite3 --column --header chinook.db
select *
from tracks
limit 1

TrackId  Name                                     AlbumId  MediaTypeId  GenreId  Composer                                   Milliseconds  Bytes     UnitPrice
-------  ---------------------------------------  -------  -----------  -------  -----------------------------------------  ------------  --------  ---------
1        For Those About To Rock (We Salute You)  1        1            1        Angus Young, Malcolm Young, Brian Johnson  343719        11170334  0.99     


In [65]:
%%script sqlite3 --column --header chinook.db
select count(*)
from tracks
where AlbumID = 1

count(*)
--------
10      


In [78]:
# 3. Do the totals in the Invoices table add up to the lineitems in the Invoice_Items table?
%%script sqlite3 --column --header chinook.db
with
inv as (
  select InvoiceId, sum(unitprice)
  from Invoice_Items
  group by InvoiceId
),
joint as (
  select *
  from inv
  join Invoices as i
  on i.invoiceid = inv.invoiceid
),
result as (
  select count(*)
  from joint
)

-- select count(distinct InvoiceId)
-- from Invoices as a
-- join result as b
select *
from joint

InvoiceId  sum(unitprice)  InvoiceId:1  CustomerId  InvoiceDate          BillingAddress                            BillingCity          BillingState  BillingCountry  BillingPostalCode  Total
---------  --------------  -----------  ----------  -------------------  ----------------------------------------  -------------------  ------------  --------------  -----------------  -----
1          1.98            1            2           2009-01-01 00:00:00  Theodor-Heuss-Straße 34                   Stuttgart                          Germany         70174              1.98 
2          3.96            2            4           2009-01-02 00:00:00  Ullevålsveien 14                          Oslo                               Norway          0171               3.96 
3          5.94            3            8           2009-01-03 00:00:00  Grétrystraat 63                           Brussels                           Belgium         1000               5.94 
4          8.91            4            14   

In [70]:
%%script sqlite3 --column --header chinook.db
select *
from Invoice_Items
limit 1

InvoiceLineId  InvoiceId  TrackId  UnitPrice  Quantity
-------------  ---------  -------  ---------  --------
1              1          2        0.99       1       


- How many unique albums are in the Tracks table?

In [ ]:
# select count(distinct albums)
# from tracks
%%script sqlite3 --column --header chinook.db
select *
from tracks
limit 10


In [ ]:
%%script sqlite3 --column --header chinook.db
select count(distinct AlbumId)
from tracks

- How many tracks are on the album with AlbumID 1?







In [ ]:
%%script sqlite3 --column --header chinook.db
select count()
from albums
limit 10

In [ ]:
%%script sqlite3 --column --header chinook.db
select *
from tracks
where AlbumId = 1

In [ ]:
%%script sqlite3 --column --header chinook.db
select count(*)
from tracks
where AlbumId = 1

- Do the totals in the Invoices table add up to the lineitems in the Invoice_Items table?

In [ ]:
%%script sqlite3 --column --header chinook.db
select *
from Invoices
limit 10

In [ ]:
%%script sqlite3 --column --header chinook.db
select *
from Invoice_Items
limit 10

In [ ]:
%%script sqlite3 --column --header chinook.db
select InvoiceId, sum(UnitPrice)
from Invoice_Items
group by InvoiceId
limit 10

In [ ]:
%%script sqlite3 --column --header chinook.db
select count(*)
from Invoices

In [ ]:
%%script sqlite3 --column --header chinook.db
select count(distinct InvoiceId)
from Invoice_Items

In [ ]:
%%script sqlite3 --column --header chinook.db
select r.EmployeeID as "EmpID", r.Firstname as "Rep", r.ReportsTo as "ReportID", s.EmployeeID as "SuperID", s.FirstName as "Supervisor"
from reps as r
inner join supervisors as s
on r.ReportsTo = s.EmployeeID

In [ ]:
%%script sqlite3 --column --header chinook.db
select InvoiceId, sum(UnitPrice) as Total
from Invoice_Items as II
group by InvoiceId
inner join Invoices
on Invoices.InvoiceId = II.Total

In [ ]:
%%script sqlite3 --column --header chinook.db
with
i as (
  select *
  from Invoices
),
ii as (
  select InvoiceId, sum(UnitPrice) as Total
  from Invoice_Items as II
  group by InvoiceId
),
iii as (
select * from i
inner join ii
on i.InvoiceId= ii.InvoiceId
where i.InvoiceId = ii.InvoiceId
order by i.InvoiceId
)

select count(*)
from iii;

## Calculating some descriptive statistics using CTEs



In [79]:
%%script sqlite3 --column --header chinook.db

WITH
  vals       as ( select unitprice as value                        from Invoice_items ),
  -- vals       as ( select Total as value                        from Invoices ),

  -- calculate descriptive statistics
  count_mean as ( select count(1) as n, max(value) as max, min(value) as min, AVG(value) as mean     from vals ),
  delta      as ( select value, n, mean, value - mean as delta from vals, count_mean ),
  delta2     as ( select power(delta, 2) as delta2             from delta ),
  delta2sum  as ( select sum(delta2) as delta2sum              from delta2 ),
  variance   as ( select delta2sum/(n-1) as variance           from delta2sum, count_mean ),
  stdev      as ( select power(variance, 0.5) as stdev         from variance ),
  desc_stats as ( select *                                     from count_mean, stdev, delta2sum, variance )

SELECT
  *
FROM
  desc_stats
;


n     max   min   mean              stdev              delta2sum         variance          
----  ----  ----  ----------------  -----------------  ----------------  ------------------
2240  1.99  0.99  1.03955357142857  0.217069229227791  105.499553571429  0.0471190502775474
